In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hurobo.utils.notebook import setup_notebook

ROOT = setup_notebook()

# MimicGen-style expansion of the source demos

`hurobo.generate.DemoGenerator` (`scripts/04_generate_dataset.py`) already does the core of **MimicGen** (Mandlekar et al., CoRL 2023): take a human demo, re-anchor it on the objects of a new layout, execute it, keep only the rollouts that LIBERO accepts. This notebook is about what is **still missing** relative to MimicGen, implemented in the separate module `hurobo.mimicgen` (existing code is imported, not changed):

| MimicGen idea | Here |
|---|---|
| nearest-neighbour source choice (pose distance of the subtask's object, pick among the `nn_k` nearest) | `SourceSelector` |
| per-subtask source choice, so segments of different demos are combined | `MixedRetargeter.plan_mixed` (approach of one demo, carry of another) |
| Gaussian action noise on the executed (recorded) actions, sigma 0.05 in simulation | `NoisyPlanExecutor` |
| success filter | already in `DemoGenerator`; here plus a filter on episode length (the evaluation allows 300 steps) |
| *not in MimicGen:* source screening | `MimicGenExpander.screen`: roll each source out alone, drop the ones that never work |

Not done: hand orientation (a `HumanDemo` holds only the TCP position, so the gripper stays top-down), motion-planned links between segments (SkillMimicGen), collision-free transit beyond the existing 20 cm hop.

**Data.** The sources are the clips of `data/e1/demos_clips` marked `usable` in its `summary.json` (real clips, not committed). If that folder is missing the stand-in demos are used and the numbers mean little. Rollouts below are few (they are a demonstration); the numbers of the longer runs are loaded from `data/e1/mimicgen/` (made by `scripts/22_mimicgen_expand.py`).

In [3]:
from hurobo.mimicgen import MimicGenConfig, MimicGenExpander, StockBaseline, load_source_demos, sample_layouts
from hurobo.real_setup import DEFAULT_MEASUREMENTS, RealSetup
from hurobo.scaled_native import ScaledNativeScene
from hurobo.utils.notebook import existing_paths

sources = existing_paths(
    {"real clips": ROOT / "data/e1/demos_clips", "stand-in": ROOT / "data/e1/standin_demos"},
    "run scripts/02_extract_demos.py or scripts/04_generate_dataset.py --stand-in 20",
)
label, demos_dir = next(iter(sources.items()))
demos, names = load_source_demos(demos_dir)
print(f"{len(demos)} source demos from {label}: {names}")

scene = ScaledNativeScene(RealSetup.load(ROOT / DEFAULT_MEASUREMENTS), seed=0)
scene.reset()
config = MimicGenConfig(nn_k=3, per_subtask=True, action_noise=0.05, seed=0)
expander = MimicGenExpander(scene, demos, names, config)

[robosuite WARNING] No private macro file found! (__init__.py:7)


[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)


[robosuite WARNING] To setup, run: python /home/eeschner/alfredo/workspace/hurobo/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


7 source demos from real clips: ['20261016_233607', '20261016_233646', '20261016_233859', '20261016_234314', '20261016_234347', '20261016_234603', '20261016_234827']


Creating LIBERO envs | suites=['libero_goal'] | n_envs(per task)=1 | init_states=True
Restricting to task_ids=[8]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Built vec env | suite=libero_goal | task_id=8 | n_envs=1
Local assets not found. Downloading from HuggingFace Hub...
Assets already downloaded at /home/eeschner/.cache/libero/assets


/home/eeschner/alfredo/workspace/hurobo/.venv/lib/python3.12/site-packages/robosuite/utils/binding_utils.py:1010: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.qpos[addr] = value


## 1. Which source demo for a new layout

MimicGen picks, per subtask, among the sources whose reference object is nearest to the new one. The two subtasks here depend on different things: the **approach** is moved rigidly with the bowl, so what matters is where the bowl is (reach limit, side of the rim); the **carry** is bent from the bowl to the plate, so what matters is the bowl-to-plate vector. Filled markers are the `nn_k` nearest.

In [4]:
from hurobo.mimicgen import source_layouts_world
from hurobo.utils.mimicgen_view import source_selection_figure

layout = sample_layouts(scene, 1, seed=3)[0]
selector = expander.selector
source_selection_figure(
    source_layouts_world(demos, expander.mapping),
    names,
    layout,
    selector.approach_distances_m(layout),
    selector.carry_distances_m(layout),
    config.nn_k,
).show()

/home/eeschner/alfredo/workspace/hurobo/.venv/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 2. Stock against MimicGen-style on the same layouts

`StockBaseline` drives `DemoGenerator.rollout` itself (uniform source, no noise, no length filter), so this is the real `scripts/04` code path. Both runs get the same layouts. Eight layouts are a demonstration, not a measurement; section 3 has the longer runs.

In [5]:
layouts = sample_layouts(scene, 8, seed=1)
stock = StockBaseline(scene, demos, names, config.retarget, seed=0).run(layouts)
mimicgen = expander.expand(layouts, None, "put the bowl on the plate")
print(stock.summary("stock (04-style)"))
print(mimicgen.summary("mimicgen-style"))

/home/eeschner/alfredo/workspace/hurobo/.venv/lib/python3.12/site-packages/robosuite/utils/binding_utils.py:1010: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  self.qpos[addr] = value


stock (04-style): 3/8 kept (38%), 314 median, 556 max steps, 1 within 300
  failures: {'timeout': 1, 'bowl not lifted': 4}
  246 kept episodes/hour (44 s wall)
mimicgen-style: 2/8 kept (25%), 260 median, 283 max steps, 2 within 300
  failures: {'timeout': 1, 'too long': 2, 'bowl not lifted': 3}
  174 kept episodes/hour (41 s wall)


In [6]:
from hurobo.utils.mimicgen_view import comparison_figure, kept_paths_figure

comparison_figure({"stock": stock, "mimicgen-style": mimicgen}).show()
kept_paths_figure(
    mimicgen, title="End-effector paths of the kept rollouts (legend: approach source / carry source)"
).show()

## 3. Longer runs

Made with `scripts/22_mimicgen_expand.py` on the usable real clips, 100 to 120 layouts each, same layout seed (so the layouts are the same across runs). What each run changes is in the run folder's `report.json` (`config`).

In [7]:
from hurobo.mimicgen import load_run_reports

runs = existing_paths(
    {path.parent.name: path for path in sorted((ROOT / "data/e1/mimicgen").glob("run_*/report.json"))},
    "run scripts/22_mimicgen_expand.py",
)
reports = {name: load_run_reports(path.parent) for name, path in runs.items()}
for name, pair in reports.items():
    for kind, report in pair.items():
        print(report.summary(f"{name} / {kind}"))

run_01_default / stock: 47/120 kept (39%), 319 median, 634 max steps, 18 within 300
  failures: {'timeout': 32, 'bowl not lifted': 40, 'plan finished without success': 1}
  138 kept episodes/hour (1228 s wall)
run_01_default / mimicgen: 15/120 kept (12%), 285 median, 300 max steps, 15 within 300
  failures: {'too long': 31, 'timeout': 40, 'bowl not lifted': 32, 'bowl dropped': 1, 'plan finished without success': 1}
  46 kept episodes/hour (1171 s wall)
run_02_nonoise / mimicgen: 19/120 kept (16%), 277 median, 298 max steps, 19 within 300
  failures: {'too long': 33, 'bowl not lifted': 32, 'timeout': 36}
  57 kept episodes/hour (1194 s wall)
run_03_nn1 / mimicgen: 5/120 kept (4%), 291 median, 297 max steps, 5 within 300
  failures: {'bowl not lifted': 25, 'timeout': 64, 'too long': 26}
  13 kept episodes/hour (1389 s wall)
run_04_screened / mimicgen: 51/100 kept (51%), 256 median, 298 max steps, 51 within 300
  failures: {'too long': 43, 'plan finished without success': 2, 'timeout': 4}

In [8]:
comparison_figure({f"{n}/{k}": r for n, pair in reports.items() for k, r in pair.items()}).show()

Per source demo: how often it was used as the approach or the carry source, and how often the rollout was kept. A success is credited to both parts of a mixed plan.

In [9]:
from hurobo.utils.mimicgen_view import source_usage_figure

for name, pair in reports.items():
    source_usage_figure(pair["mimicgen"]).update_layout(title=f"{name}: source usage").show()